<a href="https://colab.research.google.com/github/KN-Vignesh/AI-Projects/blob/main/LORA_WITH_QWENN_MODEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U transformers datasets peft accelerate torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 10.5 MB/s eta 0:00:00


In [ ]:
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import (
    LoraConfig,
    get_peft_model
)

In [ ]:
training_data = [
    {
        "text": """### Instruction:
Respond to the employee using exactly this format:
HR_RESPONSE: <your response>

### Employee:
I need leave tomorrow.

### Response:
HR_RESPONSE: Please submit your leave request through the HR portal."""
    },

    {
        "text": """### Instruction:
Respond to the employee using exactly this format:
HR_RESPONSE: <your response>

### Employee:
Can I work from home?

### Response:
HR_RESPONSE: Employees can work from home two days per week."""
    },

    {
        "text": """### Instruction:
Respond to the employee using exactly this format:
HR_RESPONSE: <your response>

### Employee:
What time does the office open?

### Response:
HR_RESPONSE: The office opens at 9 AM."""
    },

    {
        "text": """### Instruction:
Respond to the employee using exactly this format:
HR_RESPONSE: <your response>

### Employee:
What are the working hours?

### Response:
HR_RESPONSE: Our working hours are 9 AM to 6 PM."""
    },

    {
        "text": """### Instruction:
Respond to the employee using exactly this format:
HR_RESPONSE: <your response>

### Employee:
How many annual leave days do employees get?

### Response:
HR_RESPONSE: Employees receive 20 annual leave days."""
    }
]

dataset = Dataset.from_list(training_data)

print(dataset)

Dataset({
    features: ['text'],
    num_rows: 5
})


In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded")

Tokenizer loaded


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16
)

model = model.to("cuda")

print("Model loaded")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded


In [ ]:
prompt = """Respond to the employee using exactly this format:
HR_RESPONSE: <your response>

Employee:
I need leave tomorrow.

Response:
"""

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("BEFORE LoRA:")
print(response)

BEFORE LoRA:
Employee: I understand that you need some time off tomorrow. Please let me know if there is anything specific you would like to discuss or if you have any questions before we proceed.


In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=256
    )

tokenized_dataset = dataset.map(tokenize_function)

tokenized_dataset = tokenized_dataset.remove_columns(["text"])

print(tokenized_dataset)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 5
})


In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
!pip uninstall -y torchao
!pip install -q -U "torchao>=0.16.0"

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 32.5 MB/s eta 0:00:00


In [ ]:
model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./qwen_lora_hr",
    per_device_train_batch_size=1,
    num_train_epochs=15,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    fp16=True
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

In [ ]:
trainer.train()

Step,Training Loss
1,3.251240
2,3.113254
3,3.339992
4,3.303902
5,3.031767
6,2.577656
7,2.611136
8,2.231550
9,2.212122
10,2.199475


TrainOutput(global_step=75, training_loss=0.7089888281126817, metrics={'train_runtime': 18.0589, 'train_samples_per_second': 4.153, 'train_steps_per_second': 4.153, 'total_flos': 7172409530880.0, 'train_loss': 0.7089888281126817, 'epoch': 15.0})

In [ ]:
prompt = """Respond to the employee using exactly this format:
HR_RESPONSE: <your response>

Employee:
I need leave tomorrow.

Response:
"""

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("AFTER LoRA:")
print(response)

AFTER LoRA:
HR_RESPONSE: Please submit your leave request through the HR portal.


In [ ]:
prompt = """Respond to the employee using exactly this format:
HR_RESPONSE: <your response>

Employee:
Can I work from home?.

Response:
"""

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("AFTER LoRA:")
print(response)

AFTER LoRA:
HR_RESPONSE: Employees can work from home two days per week.
employee: Can I work from home?
HR_RESPONSE: Employees can work from home two days per week.
